# Credit Risk in P2P Lending: Project Guide

This notebook is the project guide for the GitHub repository. It combines the figure roadmap, data-source note, dependency note, and reproduction commands in one place.

The project studies credit-risk prediction in peer-to-peer lending: default-label construction, probability calibration, conformal uncertainty sets, distribution shift, and approval decisions.

## Data Source

The loan-level Bondora dataset is not included in this repository because the file is too large for the GitHub submission. Download it from Kaggle:

https://www.kaggle.com/datasets/marcobeyer/bondora-p2p-loans/data

Place the downloaded source file at:

```text
data/LoanData.csv
```

Generated loan-level CSV outputs are not included; the checked-in figures are the portable visual outputs.

## 1. Target Definition: Default Is A Modeling Choice

The first question is not which model wins. It is what the model is being asked to predict.

### Label choice shifts the base rate

![Label choice shifts the base rate](figures/default_label_sensitivity.png)

**Claim:** alternative default definitions change the estimand; L3 is the official default label used as the main anchor.

### Market mix changes observed risk

![Default rate by market](figures/country_default_rates.png)

**Claim:** pooled country averages can hide risk structure; the Finland pattern is a Trap3 example, not a lowest-risk conclusion.

### Vintage changes the observed population

![Entry versus 2017 vintage](figures/entry_vintage_default_rates.png)

**Claim:** time of entry matters because the platform population and observable outcomes change over calendar time.

## 2. Probability Quality: Ranking Is Not Calibration

The next step asks whether model scores can be interpreted as probabilities, not only as rankings.

### One-year calibration

![One-year calibration](figures/pod_calibration_1year.png)

**Claim:** the one-year target has its own calibration behavior; a good ranker can still misstate probability levels.

### Lifetime calibration

![Lifetime calibration](figures/pod_calibration_lifetime.png)

**Claim:** changing the horizon changes the probability scale, so one-year and lifetime risks should not be mixed.

### Brier components

![Brier components](figures/brier_score_decomposition.png)

**Claim:** Brier score combines calibration, resolution, and base-rate uncertainty; RES is subtractive and should be read as a negative contribution.

## 3. Conformal Validity: Coverage Has A Price

Conformal prediction turns scores into sets. The central tension is coverage versus decisiveness.

### Coverage equalized; width differs

![Split conformal coverage and width](figures/split_conformal_coverage_width.png)

**Claim:** conformal wrapping can align coverage near the target, but different scores pay different width and ambiguity costs.

### Marginal is not conditional

![Group conditional coverage](figures/group_conditional_coverage.png)

**Claim:** pooled coverage can look acceptable while group coverage remains uneven.

### Market shift breaks exchangeability

![Market shift diagnostic](figures/distribution_shift_diagnostic.png)

**Claim:** geography and calendar time can move the test distribution away from the calibration distribution.

### Density-ratio repair intuition

![Density-ratio view](figures/density_ratio_view_transparent.png)

**Claim:** source-like points receive lower weight and target-like points receive higher weight when calibration is reweighted toward the deployment market.

### Online control tracks drift

![Online adaptation coverage](figures/online_adaptation_coverage.png)

**Claim:** online conformal methods can steer coverage back toward 0.90 after the market moves, at the cost of wider sets.

### Label-robust validity costs width

![Label robust coverage](figures/label_robustness_coverage.png)

**Claim:** one threshold valid across multiple label definitions is more robust, but it increases set width and ambiguity.

## 4. Decisions: From Uncertainty Sets To Approval Rules

Once predictions are used for approval, the question becomes risk control under selection.

### FDR-controlled approval

![FDR-controlled approval](figures/conformal_selection_fdr.png)

**Claim:** q_FDR is a risk-appetite dial: looser FDR control increases approval power while keeping realized bad-loan share interpretable.

### TabPFN uncertainty needs conformal calibration

![TabPFN conformal set](figures/tabpfn_conformal_coverage.png)

**Claim:** native uncertainty is useful, but the conformal wrapper is what turns scores into a coverage-calibrated set.

## How To Regenerate The Figures

Run these commands from the project root. Dependencies are listed in `requirements.txt`, including `tabpfn-client` for regenerating hosted TabPFN predictions.

```powershell
python -m pip install -r requirements.txt
```

Build the conformal master file if it is missing:

```powershell
python code/bondora_credit_risk_analysis.py build-master --out data/conformal_master.csv
```

Regenerate the main slide figures:

```powershell
python code/bondora_credit_risk_analysis.py figures --outdir figures
python code/conformal_experiments.py slides --outdir figures --master data/conformal_master.csv --preds data/step5_predictions.csv
```

Regenerate hosted TabPFN predictions before the comparison figures when API access is available:

```powershell
python code/run_tabpfn_remote.py --dry-run-features
python code/run_tabpfn_remote.py --allow-remote-processing
```